# Bước 1: Kết nối (mount) Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/amazon'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Bước 2: CẢI TIẾN - XỬ LÝ DỮ LIỆU VỚI STRUCTURED SPECS PARSING

In [ ]:
import os
import pandas as pd
import numpy as np
import random
import pickle
from tqdm.notebook import tqdm
import ast

AMAZON_DIR = '/content/drive/MyDrive/amazon'
VN_META_DIRS = [
    '/content/drive/MyDrive/amazon/cleaned_mapped_metadata_1',
    '/content/drive/MyDrive/amazon/cleaned_mapped_metadata_2'
]
SAVE_DIR = '/content/drive/MyDrive/amazon/prepared_data_improved'
os.makedirs(SAVE_DIR, exist_ok=True)

NUM_CANDIDATES = 100
all_baseline_results = {}

def clean_text(val):
    if isinstance(val, list): return " ".join([str(x) for x in val])
    if isinstance(val, str):
        if val.startswith('['):
            try:
                val_list = ast.literal_eval(val)
                if isinstance(val_list, list): return " ".join([str(x) for x in val_list])
            except: pass
        return val
    if pd.isna(val): return ""
    return str(val)

def parse_specs(spec_text):
    specs = {}
    if isinstance(spec_text, list):
        for item in spec_text:
            if '::' in str(item):
                parts = str(item).split('::', 1)
                if len(parts) == 2: specs[parts[0].strip().lower()] = parts[1].strip().lower()
    elif isinstance(spec_text, str) and spec_text.startswith('['):
        try:
            items = ast.literal_eval(spec_text)
            for item in items:
                if '::' in str(item):
                    parts = str(item).split('::', 1)
                    if len(parts) == 2: specs[parts[0].strip().lower()] = parts[1].strip().lower()
        except: pass
    return specs

def get_category_from_text(breadcrumb, product_name=""):
    text = f"{str(breadcrumb)} {str(product_name)}".lower()
    if any(w in text for w in ['laptop', 'macbook', 'máy tính xách tay']): return 'laptop'
    elif any(w in text for w in ['điện thoại', 'smartphone', 'iphone', 'dtdd']): return 'smartphone'
    elif any(w in text for w in ['tivi', 'tv', 'television']): return 'television'
    elif any(w in text for w in ['tai nghe', 'headphone', 'earphone', 'airpods']): return 'headphone'
    elif any(w in text for w in ['màn hình', 'monitor']): return 'monitor'
    elif any(w in text for w in ['để bàn', 'desktop', 'pc', 'máy tính bộ']): return 'desktop'
    elif any(w in text for w in ['tablet', 'máy tính bảng', 'ipad']): return 'tablet'
    return 'other'

print("1. ĐANG XỬ LÝ KHO DỮ LIỆU VIỆT NAM...")
vn_data_list = []

for meta_dir in VN_META_DIRS:
    if not os.path.exists(meta_dir): continue
    for file in os.listdir(meta_dir):
        if file.endswith('.jsonl'):
            file_path = os.path.join(meta_dir, file)
            df_vn = pd.read_json(file_path, lines=True)
            for _, row in df_vn.iterrows():
                if pd.isna(row.get('asin')) or row.get('asin') is None: continue
                pid = str(row['product_id'])
                pname = clean_text(row.get('product_name'))
                specs = clean_text(row.get('specifications'))
                desc = clean_text(row.get('description'))
                breadcrumb = row.get('breadcrumb', '')

                category = get_category_from_text(breadcrumb, pname)

                vn_data_list.append({
                    'product_id': pid,
                    'asin': str(row['asin']).strip(),
                    'full_text': f"{pname} {specs} {desc}".lower().replace('\n', ' '),
                    'product_name': pname.lower(),
                    'parsed_specs': parse_specs(row.get('specifications')),
                    'category': category
                })

df_vn_catalog = pd.DataFrame(vn_data_list).drop_duplicates(subset=['product_id'])
vn_catalog_dict = {row['product_id']: row.to_dict() for _, row in df_vn_catalog.iterrows()}
all_vn_ids = list(vn_catalog_dict.keys())
ground_truth_map = df_vn_catalog.drop_duplicates(subset=['asin']).set_index('asin')['product_id'].to_dict()

print(f" -> Đã tải {len(all_vn_ids)} sản phẩm VN.")

print("\n2. ĐANG XỬ LÝ TRUY VẤN AMAZON...")
amz_queries = []
category_map = {'cell phones & accessories': 'smartphone', 'computers': 'desktop', 'electronics': 'other'}

for root, dirs, files in os.walk(AMAZON_DIR):
    for file in files:
        if file.endswith('_details.jsonl'):
            df_amz = pd.read_json(os.path.join(root, file), lines=True)
            for _, row in df_amz.iterrows():
                asin = str(row.get('asin', '')).strip()
                if not asin or asin == 'None' or asin not in ground_truth_map: continue

                title = clean_text(row.get('title'))
                amz_category = get_category_from_text(row.get('main_category', ''), title)

                amz_queries.append({
                    'asin': asin,
                    'query_text': f"{title} {clean_text(row.get('features'))} {clean_text(row.get('description'))}".lower().replace('\n', ' '),
                    'product_name': title.lower(),
                    'parsed_specs': parse_specs(row.get('details', {})),
                    'true_vn_id': ground_truth_map[asin],
                    'category': amz_category
                })

df_queries = pd.DataFrame(amz_queries).drop_duplicates(subset=['asin'])
print(f" -> Đã tạo {len(df_queries)} truy vấn hợp lệ.")

print("\n3. ĐANG TẠO BÀI KIỂM TRA (ÉP BUỘC ĐÚNG 100 ỨNG VIÊN & ĐẦY ĐỦ METADATA)...")
category_vn_ids = {}
for vid, vdata in vn_catalog_dict.items():
    cat = vdata.get('category', 'other')
    category_vn_ids.setdefault(cat, []).append(vid)

evaluation_dataset = []

for _, row in tqdm(df_queries.iterrows(), total=len(df_queries)):
    true_id = row['true_vn_id']
    q_category = row['category']

    same_cat_ids = [vid for vid in category_vn_ids.get(q_category, []) if vid != true_id]
    other_cat_ids = [vid for vid in all_vn_ids if vid != true_id and vid not in same_cat_ids]

    n_hard = min(int(NUM_CANDIDATES * 0.7), len(same_cat_ids))
    n_easy = (NUM_CANDIDATES - 1) - n_hard

    if n_easy > len(other_cat_ids):
        n_easy = len(other_cat_ids)
        n_hard = (NUM_CANDIDATES - 1) - n_easy

    negatives = random.sample(same_cat_ids, n_hard) + random.sample(other_cat_ids, n_easy)
    candidate_ids = [true_id] + negatives

    if len(candidate_ids) != NUM_CANDIDATES:
        continue

    random.shuffle(candidate_ids)

    # ĐÃ SỬA LỖI Ở ĐÂY: Lưu lại toàn bộ Categories và Specs cho các bước đánh giá nâng cao
    evaluation_dataset.append({
        'query_id': row['asin'],
        'query_text': row['query_text'],
        'query_name': row['product_name'],
        'query_specs': row['parsed_specs'],
        'query_category': q_category,
        'true_vn_id': true_id,
        'candidate_ids': candidate_ids,
        'candidate_texts': [vn_catalog_dict[vid]['full_text'] for vid in candidate_ids],
        'candidate_categories': [vn_catalog_dict[vid]['category'] for vid in candidate_ids],
        'candidate_specs': [vn_catalog_dict[vid]['parsed_specs'] for vid in candidate_ids],
    })

print(f"\n4. LƯU {len(evaluation_dataset)} BỘ KIỂM THỬ HOÀN CHỈNH...")
with open(os.path.join(SAVE_DIR, 'evaluation_dataset.pkl'), 'wb') as f: pickle.dump(evaluation_dataset, f)
with open(os.path.join(SAVE_DIR, 'vn_corpus.pkl'), 'wb') as f: pickle.dump(vn_catalog_dict, f)
print("HOÀN TẤT! Dữ liệu đã sẵn sàng.")

# Baseline 1: CẢI TIẾN - BM25 + Category Filter

In [ ]:
!pip install rank_bm25


In [ ]:
import pickle
import numpy as np
import os
from rank_bm25 import BM25Okapi
from tqdm.notebook import tqdm

SAVE_DIR = '/content/drive/MyDrive/amazon/prepared_data_improved'
eval_path = os.path.join(SAVE_DIR, 'evaluation_dataset.pkl')

print("Đang tải dữ liệu...")
with open(eval_path, 'rb') as f:
    raw_dataset = pickle.load(f)

# SANITY CHECK: Lọc lại một lần cuối, chỉ giữ các mẫu đủ 100 candidates
evaluation_dataset = [d for d in raw_dataset if len(d['candidate_ids']) == 100]
print(f"Đã tải và xác thực {len(evaluation_dataset)} truy vấn hợp lệ để train GCN.")

def get_ndcg_at_k(rank, k=10):
    return 1.0 / np.log2(rank + 1) if rank <= k else 0.0

hits_at_10 = 0
ndcg_at_10 = 0.0
total_queries = len(evaluation_dataset)

print(f"Bắt đầu chạy BM25 với Category Filter trên {total_queries} truy vấn...")

for data in tqdm(evaluation_dataset, total=total_queries):
    query_tokens = data['query_text'].split()
    candidate_tokens = [text.split() for text in data['candidate_texts']]

    # BM25 scoring
    bm25 = BM25Okapi(candidate_tokens)
    bm25_scores = bm25.get_scores(query_tokens)

    # Category bonus: ưu tiên cùng category
    category_scores = np.array([
        1.0 if cat == data['query_category'] else 0.0
        for cat in data['candidate_categories']
    ])

    # Hybrid: BM25 * 0.7 + Category * 0.3
    combined_scores = 0.7 * bm25_scores + 0.3 * category_scores * np.max(bm25_scores)

    ranked_indices = np.argsort(combined_scores)[::-1]
    ranked_ids = [data['candidate_ids'][i] for i in ranked_indices]

    try:
        rank = ranked_ids.index(data['true_vn_id']) + 1
        if rank <= 10:
            hits_at_10 += 1
        ndcg_at_10 += get_ndcg_at_k(rank, k=10)
    except ValueError:
        pass

print("\n" + "="*50)
print(" KẾT QUẢ BASELINE 1: BM25 + Category Filter ")
print("="*50)
if total_queries > 0:
    print(f"Hit Ratio (HR@10): {hits_at_10 / total_queries:.4f}")
    print(f"NDCG@10:           {ndcg_at_10 / total_queries:.4f}")
print("="*50)

all_baseline_results['BM25 + Category'] = {
    'HR@10': hits_at_10 / total_queries,
    'NDCG@10': ndcg_at_10 / total_queries
}

Đang tải dữ liệu...
Đã tải và xác thực 97 truy vấn hợp lệ để train GCN.
Bắt đầu chạy BM25 với Category Filter trên 97 truy vấn...


  0%|          | 0/97 [00:00<?, ?it/s]


 KẾT QUẢ BASELINE 1: BM25 + Category Filter 
Hit Ratio (HR@10): 0.4845
NDCG@10:           0.2810


# Baseline 2: CẢI TIẾN - SBERT với model mạnh hơn

In [ ]:
import pickle
import numpy as np
import torch
import os
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

SAVE_DIR = '/content/drive/MyDrive/amazon/prepared_data_improved'
eval_path = os.path.join(SAVE_DIR, 'evaluation_dataset.pkl')

print("Đang tải dữ liệu...")
with open(eval_path, 'rb') as f:
    evaluation_dataset = pickle.load(f)

# Sử dụng model mạnh hơn: paraphrase-multilingual-mpnet-base-v2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Đang tải model paraphrase-multilingual-mpnet-base-v2 trên {device}...")
model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2', device=device)

# Pre-compute VN embeddings
print("Đang pre-compute embeddings cho VN corpus...")
unique_vn_data = {}
for data in evaluation_dataset:
    for vid, vtext in zip(data['candidate_ids'], data['candidate_texts']):
        if vid not in unique_vn_data:
            unique_vn_data[vid] = vtext

vn_ids = list(unique_vn_data.keys())
vn_texts = list(unique_vn_data.values())
vn_embs_tensor = model.encode(vn_texts, batch_size=128, show_progress_bar=True, convert_to_tensor=True)
vn_emb_dict = {vid: vn_embs_tensor[i] for i, vid in enumerate(vn_ids)}

def get_ndcg_at_k(rank, k=10):
    return 1.0 / np.log2(rank + 1) if rank <= k else 0.0

hits_at_10 = 0
ndcg_at_10 = 0.0
total_queries = len(evaluation_dataset)

print(f"Bắt đầu đánh giá SBERT mpnet trên {total_queries} truy vấn...")

with torch.no_grad():
    for data in tqdm(evaluation_dataset, total=total_queries):
        query_emb = model.encode(data['query_text'], convert_to_tensor=True)
        candidate_embs = torch.stack([vn_emb_dict[vid] for vid in data['candidate_ids']])

        # Cosine similarity
        cos_scores = torch.nn.functional.cosine_similarity(query_emb.unsqueeze(0), candidate_embs).cpu().numpy()

        # Category bonus
        category_scores = np.array([
            1.0 if cat == data['query_category'] else 0.0
            for cat in data['candidate_categories']
        ])

        combined_scores = 0.7 * cos_scores + 0.3 * category_scores * np.max(cos_scores)

        ranked_indices = np.argsort(combined_scores)[::-1]
        ranked_ids = [data['candidate_ids'][i] for i in ranked_indices]

        try:
            rank = ranked_ids.index(data['true_vn_id']) + 1
            if rank <= 10:
                hits_at_10 += 1
            ndcg_at_10 += get_ndcg_at_k(rank, k=10)
        except ValueError:
            pass

print("\n" + "="*50)
print(" KẾT QUẢ BASELINE 2: SBERT mpnet + Category Filter ")
print("="*50)
if total_queries > 0:
    print(f"Hit Ratio (HR@10): {hits_at_10 / total_queries:.4f}")
    print(f"NDCG@10:           {ndcg_at_10 / total_queries:.4f}")
print("="*50)

all_baseline_results['SBERT mpnet + Category Filter'] = {
    'HR@10': hits_at_10 / total_queries,
    'NDCG@10': ndcg_at_10 / total_queries
}

Đang tải dữ liệu...
Đang tải model paraphrase-multilingual-mpnet-base-v2 trên cpu...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Đang pre-compute embeddings cho VN corpus...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Bắt đầu đánh giá SBERT mpnet trên 97 truy vấn...


  0%|          | 0/97 [00:00<?, ?it/s]


 KẾT QUẢ BASELINE 2: SBERT mpnet + Category Filter 
Hit Ratio (HR@10): 0.3918
NDCG@10:           0.2643


# Baseline 3: CẢI TIẾN - DSSM với Hard Negative Mining

In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import os
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm
import random

SAVE_DIR = '/content/drive/MyDrive/amazon/prepared_data_improved'
eval_path = os.path.join(SAVE_DIR, 'evaluation_dataset.pkl')

with open(eval_path, 'rb') as f:
    evaluation_dataset = pickle.load(f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
text_encoder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2', device=device)

# Pre-compute embeddings
print("1. Pre-compute embeddings...")
unique_vn_texts = {}
amz_data = []

for data in evaluation_dataset:
    for vid, vtext in zip(data['candidate_ids'], data['candidate_texts']):
        if vid not in unique_vn_texts:
            unique_vn_texts[vid] = vtext
    amz_data.append(data)

vn_ids = list(unique_vn_texts.keys())
vn_texts = list(unique_vn_texts.values())
vn_embs_tensor = text_encoder.encode(vn_texts, batch_size=128, show_progress_bar=True, convert_to_tensor=True)
vn_emb_dict = {vid: vn_embs_tensor[i] for i, vid in enumerate(vn_ids)}

amz_queries_text = [d['query_text'] for d in amz_data]
amz_embs_tensor = text_encoder.encode(amz_queries_text, batch_size=128, show_progress_bar=True, convert_to_tensor=True)

# Dataset với Hard Negative Mining
class TripletDatasetHardNeg(Dataset):
    def __init__(self, amz_embs, amz_data, vn_emb_dict):
        self.amz_embs = amz_embs
        self.amz_data = amz_data
        self.vn_emb_dict = vn_emb_dict

    def __len__(self):
        return len(self.amz_data)

    def __getitem__(self, idx):
        anchor_emb = self.amz_embs[idx]
        data = self.amz_data[idx]

        pos_emb = self.vn_emb_dict[data['true_vn_id']]

        # Hard negative: Lấy negative gần anchor nhất (top-k similar nhưng không đúng)
        neg_ids = [vid for vid in data['candidate_ids'] if vid != data['true_vn_id']]
        neg_embs = torch.stack([self.vn_emb_dict[vid] for vid in neg_ids])

        # Tính similarity với anchor
        similarities = torch.nn.functional.cosine_similarity(anchor_emb.unsqueeze(0), neg_embs)

        # Chọn hard negative (có similarity cao nhất)
        hard_neg_idx = torch.argmax(similarities).item()
        neg_emb = neg_embs[hard_neg_idx]

        return anchor_emb, pos_emb, neg_emb

train_dataset = TripletDatasetHardNeg(amz_embs_tensor, amz_data, vn_emb_dict)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# DSSM Model
class DSSM(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, out_dim=128):
        super(DSSM, self).__init__()
        self.amazon_tower = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, out_dim)
        )
        self.vn_tower = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, amz_emb, vn_emb):
        amz_rep = torch.nn.functional.normalize(self.amazon_tower(amz_emb), p=2, dim=1)
        vn_rep = torch.nn.functional.normalize(self.vn_tower(vn_emb), p=2, dim=1)
        return torch.sum(amz_rep * vn_rep, dim=1)

model = DSSM(input_dim=768).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

# SỬA LỖI Ở ĐÂY: Dùng MarginRankingLoss cho điểm số vô hướng (scalar)
criterion = nn.MarginRankingLoss(margin=0.2)

# Training
EPOCHS = 15
print(f"\n2. Huấn luyện DSSM trong {EPOCHS} Epochs với Hard Negatives...")
model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    for anchor, pos, neg in train_loader:
        anchor, pos, neg = anchor.to(device), pos.to(device), neg.to(device)

        optimizer.zero_grad()

        # Positive score và Negative score
        pos_score = model(anchor, pos)
        neg_score = model(anchor, neg)

        # Target = 1 có nghĩa là chúng ta muốn pos_score > neg_score
        target = torch.ones_like(pos_score).to(device)

        loss = criterion(pos_score, neg_score, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# Evaluation
print("\n3. Đánh giá...")
model.eval()

def get_ndcg_at_k(rank, k=10):
    return 1.0 / np.log2(rank + 1) if rank <= k else 0.0

hits_at_10 = 0
ndcg_at_10 = 0.0

with torch.no_grad():
    for i, data in tqdm(enumerate(amz_data), total=len(amz_data)):
        q_emb = amz_embs_tensor[i].to(device)
        c_embs = torch.stack([vn_emb_dict[vid].to(device) for vid in data['candidate_ids']])

        # Get DSSM scores
        q_rep = torch.nn.functional.normalize(model.amazon_tower(q_emb), p=2, dim=0)
        c_rep = torch.nn.functional.normalize(model.vn_tower(c_embs), p=2, dim=1)
        dssm_scores = torch.sum(q_rep.unsqueeze(0) * c_rep, dim=1).cpu().numpy()

        ranked_indices = np.argsort(dssm_scores)[::-1]
        ranked_ids = [data['candidate_ids'][j] for j in ranked_indices]

        try:
            rank = ranked_ids.index(data['true_vn_id']) + 1
            if rank <= 10:
                hits_at_10 += 1
            ndcg_at_10 += get_ndcg_at_k(rank, k=10)
        except ValueError:
            pass

print("\n" + "="*50)
print(" KẾT QUẢ BASELINE 3: DSSM + Hard Negatives ")
print("="*50)
print(f"Hit Ratio (HR@10): {hits_at_10 / len(amz_data):.4f}")
print(f"NDCG@10:           {ndcg_at_10 / len(amz_data):.4f}")
print("="*50)

all_baseline_results['DSSM + Hard Negatives'] = {
    'HR@10': hits_at_10 / total_queries,
    'NDCG@10': ndcg_at_10 / total_queries
}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


1. Pre-compute embeddings...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


2. Huấn luyện DSSM trong 15 Epochs với Hard Negatives...
Epoch 1/15 | Loss: 0.1662
Epoch 2/15 | Loss: 0.1200
Epoch 3/15 | Loss: 0.1081
Epoch 4/15 | Loss: 0.0962
Epoch 5/15 | Loss: 0.0918
Epoch 6/15 | Loss: 0.0695
Epoch 7/15 | Loss: 0.0940
Epoch 8/15 | Loss: 0.0609
Epoch 9/15 | Loss: 0.0648
Epoch 10/15 | Loss: 0.0666
Epoch 11/15 | Loss: 0.0601
Epoch 12/15 | Loss: 0.0554
Epoch 13/15 | Loss: 0.0490
Epoch 14/15 | Loss: 0.0489
Epoch 15/15 | Loss: 0.0454

3. Đánh giá...


  0%|          | 0/97 [00:00<?, ?it/s]


 KẾT QUẢ BASELINE 3: DSSM + Hard Negatives 
Hit Ratio (HR@10): 0.2371
NDCG@10:           0.1227


# Baseline 4: CẢI TIẾN - GCN với KNN threshold thấp hơn

In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm
import random

SAVE_DIR = '/content/drive/MyDrive/amazon/prepared_data_improved'
eval_path = os.path.join(SAVE_DIR, 'evaluation_dataset.pkl')

with open(eval_path, 'rb') as f:
    raw_dataset = pickle.load(f)

# SANITY CHECK: Lọc lại một lần cuối, chỉ giữ các mẫu đủ 100 candidates
evaluation_dataset = [d for d in raw_dataset if len(d['candidate_ids']) == 100]
print(f"Đã tải và xác thực {len(evaluation_dataset)} truy vấn hợp lệ để train GCN.")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
text_encoder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2', device=device)

# Pre-compute embeddings
print("1. Pre-compute embeddings...")
unique_vn_texts = {}
amz_data = []

for data in evaluation_dataset:
    for vid, vtext in zip(data['candidate_ids'], data['candidate_texts']):
        if vid not in unique_vn_texts:
            unique_vn_texts[vid] = vtext
    amz_data.append(data)

vn_ids = list(unique_vn_texts.keys())
vn_texts = list(unique_vn_texts.values())
vn_embs_tensor = text_encoder.encode(vn_texts, batch_size=128, show_progress_bar=True, convert_to_tensor=True)
vn_emb_dict = {vid: vn_embs_tensor[i] for i, vid in enumerate(vn_ids)}

amz_queries_text = [d['query_text'] for d in amz_data]
amz_embs_tensor = text_encoder.encode(amz_queries_text, batch_size=128, show_progress_bar=True, convert_to_tensor=True)

# Graph Dataset
class GraphDataset(Dataset):
    def __init__(self, amz_embs, amz_data, vn_emb_dict):
        self.amz_embs = amz_embs
        self.amz_data = amz_data
        self.vn_emb_dict = vn_emb_dict

    def __len__(self):
        return len(self.amz_data)

    def __getitem__(self, idx):
        data = self.amz_data[idx]
        q_emb = self.amz_embs[idx]
        c_embs = torch.stack([self.vn_emb_dict[vid] for vid in data['candidate_ids']])
        pos_idx = data['candidate_ids'].index(data['true_vn_id'])
        return q_emb, c_embs, pos_idx

train_loader = DataLoader(GraphDataset(amz_embs_tensor, amz_data, vn_emb_dict), batch_size=16, shuffle=True)
eval_loader = DataLoader(GraphDataset(amz_embs_tensor, amz_data, vn_emb_dict), batch_size=16, shuffle=False)

# GCN với KNN threshold = 0.3 (thay vì 0.6)
class BatchedGCN(nn.Module):
    def __init__(self, in_features=768, hidden_features=256, out_features=128, k_neighbors=10):
        super(BatchedGCN, self).__init__()
        self.k_neighbors = k_neighbors
        self.W1 = nn.Linear(in_features, hidden_features)
        self.W2 = nn.Linear(hidden_features, out_features)
        self.dropout = nn.Dropout(0.3)

    def forward(self, X):
        B, N, _ = X.size()

        # Cosine similarity
        sim_matrix = F.cosine_similarity(X.unsqueeze(2), X.unsqueeze(1), dim=3)

        # KNN với threshold = 0.3 (giảm từ 0.6)
        A = (sim_matrix > 0.3).float()

        # Self-loop + normalize
        I = torch.eye(N, device=X.device).unsqueeze(0).expand(B, -1, -1)
        A = A + I

        D = torch.sum(A, dim=2)
        D_inv_sqrt = torch.pow(D, -0.5)
        D_inv_sqrt[torch.isinf(D_inv_sqrt)] = 0.0
        D_mat_inv_sqrt = torch.diag_embed(D_inv_sqrt)

        A_norm = torch.bmm(torch.bmm(D_mat_inv_sqrt, A), D_mat_inv_sqrt)

        H1 = F.relu(self.W1(torch.bmm(A_norm, X)))
        H1 = self.dropout(H1)
        H2 = self.W2(torch.bmm(A_norm, H1))

        return F.normalize(H2, p=2, dim=2)

model = BatchedGCN(in_features=768, k_neighbors=10).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.TripletMarginLoss(margin=0.5, p=2)

# Training
EPOCHS = 15
print(f"\n2. Huấn luyện GCN với KNN threshold=0.3 trong {EPOCHS} Epochs...")
model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    for q_emb, c_embs, pos_idx in train_loader:
        q_emb, c_embs, pos_idx = q_emb.to(device), c_embs.to(device), pos_idx.to(device)
        B = q_emb.size(0)
        optimizer.zero_grad()

        X = torch.cat([q_emb.unsqueeze(1), c_embs], dim=1)
        X_out = model(X)

        anchors = X_out[:, 0, :]
        pos_indices = pos_idx + 1
        batch_indices = torch.arange(B, device=device)
        positives = X_out[batch_indices, pos_indices, :]

        # Hard negative
        neg_indices = torch.randint(1, 101, (B,), device=device)
        mask = (neg_indices == pos_indices)
        neg_indices[mask] = (neg_indices[mask] % 100) + 1
        negatives = X_out[batch_indices, neg_indices, :]

        loss = criterion(anchors, positives, negatives)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# Evaluation
print("\n3. Đánh giá GCN...")
model.eval()

def get_ndcg_at_k(rank, k=10):
    return 1.0 / np.log2(rank + 1) if rank <= k else 0.0

hits_at_10 = 0
ndcg_at_10 = 0.0

with torch.no_grad():
    for q_emb, c_embs, pos_idx in tqdm(eval_loader):
        q_emb, c_embs = q_emb.to(device), c_embs.to(device)
        B = q_emb.size(0)

        X = torch.cat([q_emb.unsqueeze(1), c_embs], dim=1)
        X_out = model(X)

        q_gcn = X_out[:, 0:1, :]
        c_gcn = X_out[:, 1:, :]

        scores = torch.sum(q_gcn * c_gcn, dim=2).cpu().numpy()
        pos_idx_np = pos_idx.numpy()

        for i in range(B):
            rank_scores = scores[i]
            true_item_index = pos_idx_np[i]
            ranked_indices = np.argsort(rank_scores)[::-1]
            rank = np.where(ranked_indices == true_item_index)[0][0] + 1
            if rank <= 10:
                hits_at_10 += 1
            ndcg_at_10 += get_ndcg_at_k(rank, k=10)

print("\n" + "="*50)
print(" KẾT QUẢ BASELINE 4: GCN (KNN=0.3) ")
print("="*50)
print(f"Hit Ratio (HR@10): {hits_at_10 / len(amz_data):.4f}")
print(f"NDCG@10:           {ndcg_at_10 / len(amz_data):.4f}")
print("="*50)

all_baseline_results['GCN'] = {
    'HR@10': hits_at_10 / total_queries,
    'NDCG@10': ndcg_at_10 / total_queries
}

Đã tải và xác thực 97 truy vấn hợp lệ để train GCN.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


1. Pre-compute embeddings...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


2. Huấn luyện GCN với KNN threshold=0.3 trong 15 Epochs...
Epoch 1/15 | Loss: 0.4997
Epoch 2/15 | Loss: 0.5006
Epoch 3/15 | Loss: 0.5010
Epoch 4/15 | Loss: 0.5006
Epoch 5/15 | Loss: 0.5024
Epoch 6/15 | Loss: 0.5015
Epoch 7/15 | Loss: 0.4993
Epoch 8/15 | Loss: 0.5161
Epoch 9/15 | Loss: 0.4998
Epoch 10/15 | Loss: 0.4965
Epoch 11/15 | Loss: 0.4986
Epoch 12/15 | Loss: 0.4918
Epoch 13/15 | Loss: 0.4961
Epoch 14/15 | Loss: 0.4909
Epoch 15/15 | Loss: 0.4995

3. Đánh giá GCN...


  0%|          | 0/7 [00:00<?, ?it/s]


 KẾT QUẢ BASELINE 4: GCN (KNN=0.3) 
Hit Ratio (HR@10): 0.1134
NDCG@10:           0.0483


# Baseline 5: CẢI TIẾN - HYBRID BM25 + SBERT + Attribute Matching

In [ ]:
import pickle
import numpy as np
import torch
import os
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from tqdm.notebook import tqdm

SAVE_DIR = '/content/drive/MyDrive/amazon/prepared_data_improved'
eval_path = os.path.join(SAVE_DIR, 'evaluation_dataset.pkl')

with open(eval_path, 'rb') as f:
    raw_dataset = pickle.load(f)

# SANITY CHECK: Lọc lại một lần cuối, chỉ giữ các mẫu đủ 100 candidates
evaluation_dataset = [d for d in raw_dataset if len(d['candidate_ids']) == 100]
print(f"Đã tải và xác thực {len(evaluation_dataset)} truy vấn hợp lệ để train GCN.")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2', device=device)

# Pre-compute embeddings
print("Pre-compute VN embeddings...")
unique_vn_data = {}
for data in evaluation_dataset:
    for vid, vtext in zip(data['candidate_ids'], data['candidate_texts']):
        if vid not in unique_vn_data:
            unique_vn_data[vid] = vtext

vn_ids = list(unique_vn_data.keys())
vn_texts = list(unique_vn_data.values())
vn_embs_tensor = model.encode(vn_texts, batch_size=128, show_progress_bar=True, convert_to_tensor=True)
vn_emb_dict = {vid: vn_embs_tensor[i] for i, vid in enumerate(vn_ids)}

def compute_attribute_match(q_specs, c_specs):
    """Tính điểm match giữa specs của query và candidate"""
    if not q_specs or not c_specs:
        return 0.0

    matches = 0
    total = 0

    # Các key quan trọng cần match
    important_keys = ['công nghệ cpu', 'chip xử lý', 'ram', 'ổ cứng', 'brand', 'thương hiệu']

    for key in important_keys:
        if key in q_specs and key in c_specs:
            total += 1
            if q_specs[key] == c_specs[key]:
                matches += 1

    return matches / max(total, 1)

def get_ndcg_at_k(rank, k=10):
    return 1.0 / np.log2(rank + 1) if rank <= k else 0.0

hits_at_10 = 0
ndcg_at_10 = 0.0
total_queries = len(evaluation_dataset)

print(f"\nChạy Hybrid (BM25 + SBERT + Attribute) trên {total_queries} truy vấn...")

for data in tqdm(evaluation_dataset, total=total_queries):
    # BM25 scores
    query_tokens = data['query_text'].split()
    candidate_tokens = [text.split() for text in data['candidate_texts']]
    bm25 = BM25Okapi(candidate_tokens)
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_scores = np.array(bm25_scores) / (np.max(bm25_scores) + 1e-6)

    # SBERT scores
    query_emb = model.encode(data['query_text'], convert_to_tensor=True)
    candidate_embs = torch.stack([vn_emb_dict[vid] for vid in data['candidate_ids']])
    sbert_scores = torch.nn.functional.cosine_similarity(query_emb.unsqueeze(0), candidate_embs).cpu().numpy()

    # Attribute match scores
    q_specs = data.get('query_specs', {})
    attr_scores = np.array([
        compute_attribute_match(q_specs, c_specs)
        for c_specs in data.get('candidate_specs', [])
    ])

    # Category bonus
    category_scores = np.array([
        1.0 if cat == data['query_category'] else 0.0
        for cat in data['candidate_categories']
    ])

    # Hybrid: BM25*0.2 + SBERT*0.3 + Attr*0.3 + Category*0.2
    combined_scores = (
        0.2 * bm25_scores +
        0.3 * sbert_scores +
        0.3 * attr_scores +
        0.2 * category_scores
    )

    ranked_indices = np.argsort(combined_scores)[::-1]
    ranked_ids = [data['candidate_ids'][i] for i in ranked_indices]

    try:
        rank = ranked_ids.index(data['true_vn_id']) + 1
        if rank <= 10:
            hits_at_10 += 1
        ndcg_at_10 += get_ndcg_at_k(rank, k=10)
    except ValueError:
        pass

print("\n" + "="*50)
print(" KẾT QUẢ BASELINE 5: HYBRID (BM25 + SBERT + Attribute) ")
print("="*50)
if total_queries > 0:
    print(f"Hit Ratio (HR@10): {hits_at_10 / total_queries:.4f}")
    print(f"NDCG@10:           {ndcg_at_10 / total_queries:.4f}")
print("="*50)

all_baseline_results['HYBRID (BM25 + SBERT + Attribute)'] = {
    'HR@10': hits_at_10 / total_queries,
    'NDCG@10': ndcg_at_10 / total_queries
}

Đã tải và xác thực 97 truy vấn hợp lệ để train GCN.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pre-compute VN embeddings...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]


Chạy Hybrid (BM25 + SBERT + Attribute) trên 97 truy vấn...


  0%|          | 0/97 [00:00<?, ?it/s]


 KẾT QUẢ BASELINE 5: HYBRID (BM25 + SBERT + Attribute) 
Hit Ratio (HR@10): 0.3918
NDCG@10:           0.2452


# So sánh tất cả Baselines

In [ ]:
import matplotlib.pyplot as plt

# TỰ ĐỘNG LẤY DỮ LIỆU TỪ QUÁ TRÌNH CHẠY
results = all_baseline_results

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

methods = list(results.keys())
hr_scores = [results[m]['HR@10'] for m in methods]
ndcg_scores = [results[m]['NDCG@10'] for m in methods]

x = range(len(methods))

# Biểu đồ HR@10
axes[0].bar(x, hr_scores, color='steelblue')
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods, rotation=45, ha='right')
axes[0].set_ylabel('HR@10')
axes[0].set_title('Hit Ratio @10')
axes[0].set_ylim(0, 1.1)
for i, v in enumerate(hr_scores):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)

# Biểu đồ NDCG@10
axes[1].bar(x, ndcg_scores, color='darkorange')
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods, rotation=45, ha='right')
axes[1].set_ylabel('NDCG@10')
axes[1].set_title('NDCG@10')
axes[1].set_ylim(0, 1.1)
for i, v in enumerate(ndcg_scores):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/amazon/baseline_comparison.png', dpi=150)
plt.show()

print("\nBiểu đồ đã được vẽ tự động từ hệ thống!")